# 2) Silver Layer
## Barcelona Urban Intelligence Assistant
### Advanced Data Processing and Analysis

The bronze layer gave the datasets in different formats, languages, and naming conventions. The silver layer has to make sure they are all in the same format so the gold layer can join the different datasets.
An important task here was that the names of each district had to be made identical across every dataset. The IRIS complaint dataset writes this district as ‘Sarrià-St. Gervasi’, the electric vehicle file as ‘SARRIA ST. GERVASI’, and the accident file as ‘Sarrià-Sant Gervasi’.

The silver layer performs five transformations:
- District name standardisation: Without name standardisation, every join in the gold layer produces the wrong row counts and wrong aggregations
- Date reconstruction (YYYY-MM-DD format): The DuckDB’s date arithmetic requires the ISO formal, and the wrong formal means that all of the resolution times compute as NULL
- Catalan to English translation: Enabling consistent category grouping in ML features and the RAG retrieval
- PII masking: Complying with GDPR, free-text complaints can contain names, phone numbers, and email addresses
- Type casting: Ensuring avg(), sum(), date_diff() produce the correct results


In [ ]:
import duckdb
import pandas as pd
import numpy as np
import re
import os

os.chdir(r"C:\Users\zarah\Downloads\Spring2026\Advanced Data\Final Project")
con = duckdb.connect("barcelona_urban.db")
print("Connected to barcelona_urban.db")


Connected to barcelona_urban.db


## Helper functions

Three user-defined functions:
- `clean_district()` — the most critical function in the project. Every Gold join depends on it.
- `translate_cause()` — maps 18 Catalan accident causes to English. Required for ML feature consistency.
- `translate_type()` — maps IRIS complaint type codes to English labels.


In [ ]:
# Helping functions
# District name standardizer
def clean_district(name: str) -> str:
    try:
        if name is None:
            return "UNKNOWN"
        if str(name).strip().lower() in ("", "nan", "none", "null", "-1"):
            return "UNKNOWN"
        name = str(name).upper().strip()
        for src, tgt in [("À","A"),("È","E"),("Í","I"),("Ó","O"),("Ú","U"),
                         ("Á","A"),("É","E"),("Ì","I"),("Ò","O"),("Ù","U"),
                         ("Ï","I"),("Ü","U"),("Ñ","N"),("Ç","C"),
                         ("·"," "),("-"," ")]:
            name = name.replace(src, tgt)
        name = name.replace("ST.", "SANT")
        name = " ".join(name.split())
        return name.strip()
    except Exception:
        return "UNKNOWN"

# tests, have to pass before continuing, and when there is a 'None', "NaN", "nan" value found it gets replaced by UNKNOWN
assert clean_district("Sarrià-St. Gervasi") == "SARRIA SANT GERVASI"
assert clean_district("SANT MARTI") == "SANT MARTI"
assert clean_district(None) == "UNKNOWN"
assert clean_district("NaN") == "UNKNOWN"
assert clean_district("nan") == "UNKNOWN"
assert clean_district("") == "UNKNOWN"
print("clean_district() registered and all tests passed")

# accident cause translator (Catalan → English)
CAUSE_MAP = {
    "No respectar distàncies" : "Not keeping safe distance",
    "No respectar distancies" : "Not keeping safe distance",
    "Distracció del conductor" : "Driver distraction",
    "Distraccio del conductor": "Driver distraction",
    "No respectar senyals de trànsit" : "Ignoring traffic signals",
    "No respectar la prioritat de pas" : "Failing to yield",
    "Velocitat inadequada": "Inappropriate speed",
    "Girar indegudament" : "Improper turning",
    "Canvi de carril indegut": "Improper lane change",
    "Alcoholèmia" : "Drink driving",
    "Alcoholemia"  : "Drink driving",
    "Drogues" : "Drug impairment",
    "Avaria" : "Vehicle breakdown",
    "Il·luminació deficient" : "Poor lighting",
    "Mal estat de la via" : "Poor road condition",
    "Avançament indegut"  : "Improper overtaking",
    "Invasió del sentit contrari" : "Wrong-way driving",
    "Altres causes"  : "Other causes",
    "Causa desconeguda" : "Unknown cause",
}

def translate_cause(cause: str) -> str:
    try:
        if cause is None or str(cause).strip() in ("", "nan"):
            return "Unknown"
        return CAUSE_MAP.get(str(cause).strip(), str(cause).strip())
    except Exception:
        return "Unknown"

print("translate_cause() registered")

# complaint type translator (Catalan → English)
TYPE_MAP = {
    "INCIDENCIA" : "Incident",
    "QUEIXA": "Complaint",
    "SUGGERIMENT" : "Suggestion",
    "CONSULTA" : "Enquiry",
    "FELICITACIO" : "Compliment",
    "RECLAMACIO" : "Claim",
    "PETICIO" : "Request",
}

def translate_type(t: str) -> str:
    try:
        if t is None or str(t).strip() in ("", "nan"):
            return "Unknown"
        return TYPE_MAP.get(str(t).strip().upper(), str(t).strip())
    except Exception:
        return "Unknown"

print("translate_type() registered")

# Reconnect to clear any previous UDF registrations (makes cell safe to re-run)
con.close()
con = duckdb.connect("barcelona_urban.db")

# Register Python functions as DuckDB scalar UDFs so SQL can call them
# null_handling='special' passes SQL NULLs as Python None instead of short-circuiting to NULL
# This ensures clean_district(NULL) returns 'UNKNOWN' instead of NULL
con.create_function('clean_district',  clean_district,  ['VARCHAR'], 'VARCHAR', null_handling='special')
con.create_function('translate_cause', translate_cause, ['VARCHAR'], 'VARCHAR', null_handling='special')
con.create_function('translate_type',  translate_type,  ['VARCHAR'], 'VARCHAR', null_handling='special')
print('UDFs registered with DuckDB connection')


clean_district() registered and all tests passed
translate_cause() registered
translate_type() registered
UDFs registered with DuckDB connection


## Silver 1: Complaints (IRIS)

The IRIS file stores dates as three separate integer columns: ‘DIA_DATA_ALTA’ (day), ‘MES_DATA_ALTA’ (month), and ‘ANY_DATA_ALTA’ (year). We put these dates into an ISO date (‘YYYY-MM-DD’). The gap between the opening and closing date of the complaint, resolution time, is the most important metric in the third research question. Since it is a part of the individual complaint we will compute it in the silver layer, and then the gold layer will average it.


In [ ]:
con.execute("""
CREATE OR REPLACE TABLE silver.complaints AS
SELECT
    CAST(FITXA_ID AS BIGINT)                                         AS complaint_id,
    translate_type(TIPUS)                                            AS complaint_type,
    COALESCE(NULLIF(TRIM(AREA),  ''), 'Unknown')                     AS area,
    COALESCE(NULLIF(TRIM(ELEMENT),''), 'Unknown')                    AS element,
    COALESCE(NULLIF(TRIM(DETALL), ''), 'Unknown')                    AS detail,

    -- Build complaint date: must be YYYY-MM-DD for DuckDB TRY_CAST
    TRY_CAST(
        LPAD(ANY_DATA_ALTA, 4, '0')     || '-' ||
        LPAD(MES_DATA_ALTA, 2, '0')     || '-' ||
        LPAD(DIA_DATA_ALTA, 2, '0')
    AS DATE)                                                         AS complaint_date,

    -- Build closing date: YYYY-MM-DD
    TRY_CAST(
        LPAD(ANY_DATA_TANCAMENT, 4, '0') || '-' ||
        LPAD(MES_DATA_TANCAMENT, 2, '0') || '-' ||
        LPAD(DIA_DATA_TANCAMENT, 2, '0')
    AS DATE)                                                         AS closing_date,

    -- Resolution days: difference between open and close date
    CASE
        WHEN TRY_CAST(ANY_DATA_ALTA AS INT) IS NOT NULL
         AND TRY_CAST(ANY_DATA_TANCAMENT AS INT) IS NOT NULL
        THEN DATE_DIFF('day',
            TRY_CAST(
                LPAD(ANY_DATA_ALTA, 4, '0') || '-' ||
                LPAD(MES_DATA_ALTA, 2, '0') || '-' ||
                LPAD(DIA_DATA_ALTA, 2, '0') AS DATE),
            TRY_CAST(
                LPAD(ANY_DATA_TANCAMENT, 4, '0') || '-' ||
                LPAD(MES_DATA_TANCAMENT, 2, '0') || '-' ||
                LPAD(DIA_DATA_TANCAMENT, 2, '0') AS DATE))
        ELSE NULL
    END                                                              AS resolution_days,

    -- Year from complaint date (for filtering / aggregation)
    TRY_CAST(ANY_DATA_ALTA AS INT)                                   AS complaint_year,
    TRY_CAST(MES_DATA_ALTA AS INT)                                   AS complaint_month,

    -- District and neighborhood — standardized
    clean_district(DISTRICTE)                                        AS district,
    COALESCE(NULLIF(TRIM(BARRI), ''), 'Unknown')                     AS neighborhood,
    COALESCE(NULLIF(TRIM(SUPORT), ''), 'Unknown')                    AS channel

FROM bronze.iris_raw
WHERE FITXA_ID IS NOT NULL
  AND FITXA_ID NOT IN ('', 'nan', 'None')
""")

n = con.execute("SELECT COUNT(*) FROM silver.complaints").fetchone()[0]
print(f"silver.complaints: {n:,} rows")
display(con.execute("""
    SELECT district, COUNT(*) AS n,
           ROUND(AVG(resolution_days),1) AS avg_days,
           MIN(complaint_date) AS earliest,
           MAX(closing_date) AS latest
    FROM silver.complaints
    GROUP BY district ORDER BY n DESC
""").df())


silver.complaints: 287,304 rows


,district,n,avg_days,earliest,latest
0,UNKNOWN,89936,9.8,2023-12-14,2025-12-31
1,EIXAMPLE,34608,7.4,2024-06-09,2025-12-31
2,SANT MARTI,32321,6.8,2024-02-27,2025-12-31
3,SANTS MONTJUIC,21985,11.2,2024-04-07,2025-12-31
4,HORTA GUINARDO,20151,9.8,2024-04-24,2025-12-31
5,CIUTAT VELLA,19874,8.7,2024-05-31,2025-12-31
6,NOU BARRIS,17067,9.6,2024-01-23,2025-12-31
7,SANT ANDREU,16358,8.3,2023-10-25,2025-12-31
8,GRACIA,15007,8.5,2024-05-05,2025-12-31
9,SARRIA SANT GERVASI,12941,10.1,2024-06-12,2025-12-31


In [ ]:
n_total   = con.execute("SELECT COUNT(*) FROM silver.complaints").fetchone()[0]
n_unknown = con.execute("""
    SELECT COUNT(*) FROM silver.complaints
    WHERE district = 'UNKNOWN' OR district IS NULL
""").fetchone()[0]
print(f"{n_unknown:,} / {n_total:,} = {100*n_unknown/n_total:.1f}% unassigned")


89,936 / 287,304 = 31.3% unassigned


Interpretation: The largest group in silver.complaints has no district assignment (UNKNOWN/null), this is because the district is an optional field when filing a complaint. In total (89892/ 287,304) * 100 = 31.39% of the complaints is unassigned to a district. This exclusion of these complaints not belonging to an assigned district can introduce a mild selection bias if certain complaint types are disproportionately submitted without a district, this is a limitation of the IRIS dataset.

## PII Masking: GDPR compliance

PII:
- Names: e.g. "Call Mr. Garcia about the broken lamp on my street"
- Phone numbers: e.g. "You can reach me at 634-221-098"
- Email addresses: e.g. "Please reply to maria.lopez@gmail.com"

Under GDPR article 5, the data minimisation principle, you may only keep personal data for as long as it is strictly necessary for the purpose it was collected for (European Data Protection Supervisor [EDPS], n.d.). When we analyse complaint patterns across different districts we don’t need to know who filed the complaint, we only need the complaint type, district, and resolution time. If we don’t mask PII we are in violation of GDPR, any accident data leak exposes real people, and La Salle in this case as the data controller could face regulatory consequences, and the dataset can’t ethically be shared with other people.

In [ ]:
# PII masking: GDPR compliant scrubbing
# Removes personal information from free-text fields before we do anya analysis

import re

def scrub_pii(text: str) -> str:
    if text is None:
        return text
    text_str = str(text).strip()
    if text_str.lower() in ('', 'nan', 'none', 'null'):
        return text

    # Email addresses:
    text_str = re.sub(
        r'\b[A-Za-z0-9._%+\-]+@[A-Za-z0-9.\-]+\.[A-Za-z]{2,}\b',
        '[EMAIL]',
        text_str
    )

    # Spanish/Catalan phone numbers:
    text_str = re.sub(
        r'\b(\+34|0034)?[\s\-]?[6789]\d{2}[\s\-]?\d{3}[\s\-]?\d{3}\b',
        '[PHONE]',
        text_str
    )
    # Also landlines formatted as 93x-xxx-xx-xx (Barcelona area code 93)
    text_str = re.sub(
        r'\b9[0-9][\s\-]?\d{3}[\s\-]?\d{2}[\s\-]?\d{2}\b',
        '[PHONE]',
        text_str
    )

    # honorific + name (e.g. "Sr. Garcia", "Sra. Lopez")
    text_str = re.sub(
        r'\b(Sr\.|Sra\.|Dr\.|Dra\.|Mr\.|Mrs\.|Ms\.)\s+[A-ZÁÉÍÓÚÀÈÏÜÇ][a-záéíóúàèïüç]+\b',
        '[NAME]',
        text_str
    )

    return text_str

# Tests
test_cases = [
    ("Call me at 634221098 please",        True,   "[PHONE]"),
    ("Email: maria@gmail.com",             True,   "[EMAIL]"),
    ("Ask Sr. Garcia about this",          True,   "[NAME]"),
    ("The lamp on Carrer Major is broken", False,  None),
    (None,                                 False,  None),
]
print("scrub_pii() tests:")
for text, should_change, expected_tag in test_cases:
    result = scrub_pii(text)
    if should_change:
        assert expected_tag in str(result), f"FAIL: expected {expected_tag} in result for: {text}"
        print(f"'{text[:40]}...' → '{str(result)[:40]}...'")
    else:
        print(f"No PII in: '{text}' → unchanged")
print("All scrub_pii() tests passed")

scrub_pii() tests:
'Call me at 634221098 please...' → 'Call me at[PHONE] please...'
'Email: maria@gmail.com...' → 'Email: [EMAIL]...'
'Ask Sr. Garcia about this...' → 'Ask [NAME] about this...'
No PII in: 'The lamp on Carrer Major is broken' → unchanged
No PII in: 'None' → unchanged
All scrub_pii() tests passed


### Applying PII Masking to IRIS Complaints

We apply `scrub_pii()` to the `DETALL` column, the free-text field where citizens describe their complaint in their own words.

We obviously don't mask:
- `DISTRICTE` (district name): geographic area, not personal data
- `TIPUS` (complaint type): categorical, not personal
- `FITXA_ID` (complaint ID): an internal reference number, not a personal identifier

Before masking example:
- *"Vull que es repari el fanal del carrer. Podeu contactar-me al 634221098 o a sr.garcia@correo.es"*

After masking example:
- *"Vull que es repari el fanal del carrer. Podeu contactar-me al [PHONE] o a [EMAIL]"*

The complaint is now usable for analysis (district, complaint type, resolution time), and the personal contact details are gone.


In [ ]:
try:
    con.create_function('scrub_pii', scrub_pii, ['VARCHAR'], 'VARCHAR', null_handling='special')
except duckdb.CatalogException:
    pass  # Already registered; safe to continue


con.execute("""
CREATE OR REPLACE TABLE silver.complaints_masked AS
SELECT
    complaint_id,
    complaint_type,
    area,
    element,
    scrub_pii(detail)   AS detail,
    complaint_date,
    closing_date,
    resolution_days,
    complaint_year,
    complaint_month,
    district,
    neighborhood,
    channel
FROM silver.complaints
""")

n_masked = con.execute("SELECT COUNT(*) FROM silver.complaints_masked").fetchone()[0]
print(f"silver.complaints_masked: {n_masked:,} rows created")


silver.complaints_masked: 287,304 rows created


In [ ]:
# Applying PII masking to IRIS complaints
# Creating silver.complaints_masked as a seperate table.
# The Gold layer and ML pipeline read only from silver.complaints_masked.

# Showing before/after examples to verify masking works
print("PII masking before & after examples\n")

# Pulling a sample of rows that likely contain PII by looking for @ symbol or phone patterns
sample_with_pii = con.execute("""
    SELECT complaint_id, district, complaint_type, detail
    FROM silver.complaints
    WHERE detail LIKE '%@%'
       OR regexp_matches(detail, '[6789][0-9]{8}')
    LIMIT 5
""").df()

if len(sample_with_pii) > 0:
    print("Rows with potential PII found:")
    for _, row in sample_with_pii.iterrows():
        original = str(row['detail'])
        masked   = scrub_pii(original)
        if original != masked:
            print(f"  BEFORE: {original[:80]}")
            print(f"  AFTER:  {masked[:80]}")
            print()
else:
    print("No obvious PII patterns found in sample (emails/phone numbers).")
    print("The scrub_pii() function will still process all rows as a safeguard.")

# Applying masking and creating the masked silver table
print("\nCreating silver.complaints_masked...")
con.execute("""
CREATE OR REPLACE TABLE silver.complaints_masked AS
SELECT
    complaint_id,
    complaint_type,
    area,
    element,
    scrub_pii(detail)   AS detail,
    complaint_date,
    closing_date,
    resolution_days,
    complaint_year,
    complaint_month,
    district,
    neighborhood,
    channel
FROM silver.complaints
""")

n_masked = con.execute("SELECT COUNT(*) FROM silver.complaints_masked").fetchone()[0]
print(f"silver.complaints_masked: {n_masked:,} rows created")

PII masking before & after examples

No obvious PII patterns found in sample (emails/phone numbers).
The scrub_pii() function will still process all rows as a safeguard.

Creating silver.complaints_masked...
silver.complaints_masked: 287,304 rows created


Interpretation: All the downstream gold tables and ML features read from silver.complaints_masked and never from silver.complaints or bronze.iris_raw.

In [ ]:
#PII masking validation checks to confirm if the masking went correctly
print("PII masking validation\n")

# Row count must be identical
n_original = con.execute("SELECT COUNT(*) FROM silver.complaints").fetchone()[0]
n_masked   = con.execute("SELECT COUNT(*) FROM silver.complaints_masked").fetchone()[0]
assert n_original == n_masked, f"Row count mismatch! original={n_original}, masked={n_masked}"
print(f"Row count preserved: {n_masked:,} rows (no rows dropped by masking)")

# Count how many rows had PII replaced
n_emails  = con.execute("""
    SELECT COUNT(*) FROM silver.complaints_masked WHERE detail LIKE '%[EMAIL]%'
""").fetchone()[0]
n_phones  = con.execute("""
    SELECT COUNT(*) FROM silver.complaints_masked WHERE detail LIKE '%[PHONE]%'
""").fetchone()[0]
n_names   = con.execute("""
    SELECT COUNT(*) FROM silver.complaints_masked WHERE detail LIKE '%[NAME]%'
""").fetchone()[0]
print(f"\nPII instances replaced:")
print(f"  [EMAIL] tags inserted: {n_emails:,} rows")
print(f"  [PHONE] tags inserted: {n_phones:,} rows")
print(f"  [NAME]  tags inserted: {n_names:,}  rows")

# No @ symbols should remain in detail column
n_remaining_emails = con.execute("""
    SELECT COUNT(*) FROM silver.complaints_masked
    WHERE detail LIKE '%@%'
""").fetchone()[0]
if n_remaining_emails == 0:
    print(f"\nNo raw @ symbols remain in detail column")
else:
    print(f"\n  {n_remaining_emails} rows may still contain email addresses: review patterns")

# Show 3 masked rows for visual verification
print("\nSpot-check — 3 rows with masking applied (detail column):")
display(con.execute("""
    SELECT complaint_id, district, detail
    FROM silver.complaints_masked
    WHERE detail LIKE '%[EMAIL]%' OR detail LIKE '%[PHONE]%' OR detail LIKE '%[NAME]%'
    LIMIT 3
""").df())

print("\n PII masking validation complete.")
print("GDPR note: silver.complaints_masked is the ONLY complaints table")
print("that should be used in the Gold layer, ML analysis, and the RAG system.")


PII masking validation

Row count preserved: 287,304 rows (no rows dropped by masking)

PII instances replaced:
  [EMAIL] tags inserted: 0 rows
  [PHONE] tags inserted: 56 rows
  [NAME]  tags inserted: 0  rows

No raw @ symbols remain in detail column

Spot-check — 3 rows with masking applied (detail column):


,complaint_id,district,detail
0,38757406,UNKNOWN,Desacord amb el procediment del telèfon[PHONE]
1,38759365,UNKNOWN,Desacord amb l'atenció al telèfon[PHONE] (010 ...
2,38757622,UNKNOWN,Desacord amb el procediment del telèfon[PHONE]



 PII masking validation complete.
GDPR note: silver.complaints_masked is the ONLY complaints table
that should be used in the Gold layer, ML analysis, and the RAG system.


## Silver 2: traffic accidents

In [ ]:
con.execute("""
CREATE OR REPLACE TABLE silver.accidents AS
SELECT
    "Número_expedient" AS accident_id,
    clean_district(Nom_districte)  AS district,
    COALESCE(NULLIF(TRIM(Nom_barri), ''), 'Unknown') AS neighborhood,
    translate_cause("Causa_conductor")  AS accident_cause,
    TRY_CAST(Nk_Any  AS INT) AS year,
    TRY_CAST(Mes_any AS INT) AS month,
    TRY_CAST(Dia_mes AS INT) AS day,
    TRY_CAST(Hora_dia AS INT)  AS hour,
    COALESCE(NULLIF(TRIM(Descripcio_torn), ''), 'Unknown')  AS shift,
    COALESCE(NULLIF(TRIM(Descripcio_dia_setmana),''), 'Unknown')  AS weekday,
    TRY_CAST(Longitud_WGS84 AS DOUBLE) AS longitude,
    TRY_CAST(Latitud_WGS84  AS DOUBLE) AS latitude,

    TRY_CAST(
        CAST(Nk_Any AS VARCHAR)                    || '-' ||
        LPAD(CAST(Mes_any AS VARCHAR), 2, '0')     || '-' ||
        LPAD(CAST(Dia_mes AS VARCHAR), 2, '0')
    AS DATE)                                                         AS accident_date

FROM bronze.accidents_raw
WHERE Nom_districte IS NOT NULL
  AND TRIM(Nom_districte) NOT IN ('', '-1')
""")

n = con.execute("SELECT COUNT(*) FROM silver.accidents").fetchone()[0]
print(f"silver.accidents: {n:,} rows")
display(con.execute("""
    SELECT district, COUNT(*) AS accidents,
           COUNT(DISTINCT accident_cause) AS distinct_causes
    FROM silver.accidents
    GROUP BY district ORDER BY accidents DESC
""").df())


silver.accidents: 8,072 rows


,district,accidents,distinct_causes
0,EIXAMPLE,2009,16
1,SANT MARTI,1097,16
2,SANTS MONTJUIC,963,16
3,SARRIA SANT GERVASI,828,16
4,HORTA GUINARDO,630,16
5,LES CORTS,624,15
6,SANT ANDREU,587,15
7,NOU BARRIS,487,16
8,GRACIA,428,16
9,CIUTAT VELLA,419,16


## Silver 3: Air quality

Each row in the air quality dataset is one day at one station with 24 hourly measurements (H01-H24) and 24 validity flags as separate columns (V01-V24).

To answer the question “What was the average NO₂ level in Eixample in 2025?” we have to do the following:
- Filter the 2 pollutants that align with our framework metrics, NO₂ for the Paris Agreement and PM₁₀ as the EU health standard.
- Average only the valid hours that is V = ‘V’ in the dataset and V = ‘N’ should be excluded, otherwise the average is wrong.
- Join the monthly air quality datasets to the station metadata file so we can learn which district has which station.
- Aggregate the daily averages to monthly district averages.

Three districts have no PM₁₀ data, these districts do have NO₂ instruments but no PM₁₀ sensor, this is imputed with the city mean later at the gold layer.

In [ ]:
# Loading raw air quality
aq_df = con.execute("SELECT * FROM bronze.airquality_raw").df()
print(f"Raw air quality rows: {len(aq_df):,}")
print("Pollutant codes present:", sorted(aq_df["CODI_CONTAMINANT"].dropna().unique().tolist()))

# Filter for NO2 (code 8) and PM10 (code 10)
aq_df["CODI_CONTAMINANT"] = pd.to_numeric(aq_df["CODI_CONTAMINANT"], errors="coerce")
aq_filtered = aq_df[aq_df["CODI_CONTAMINANT"].isin([8, 10])].copy()
print(f"After filtering NO2+PM10: {len(aq_filtered):,} rows")


Raw air quality rows: 22,960
Pollutant codes present: ['1', '10', '101', '106', '107', '108', '109', '11', '110', '111', '112', '114', '12', '14', '22', '6', '7', '8', '9', '901', '995', '996', '997', '998', '999']
After filtering NO2+PM10: 4,592 rows


In [ ]:
# Computing daily average from valid hourly readings
# H01-H24 = measured values, V01-V24 = validation flag ('V' = valid, 'N' = invalid)
h_cols = [f"H{i:02d}" for i in range(1, 25)]
v_cols = [f"V{i:02d}" for i in range(1, 25)]

# Converting H columns to numeric
for c in h_cols:
    aq_filtered[c] = pd.to_numeric(aq_filtered[c], errors="coerce")

# Building boolean: true where V flag = 'V' (valid reading)
valid_mask = pd.DataFrame(
    {f"H{i:02d}": aq_filtered[f"V{i:02d}"].str.strip().eq("V") for i in range(1, 25)},
    index=aq_filtered.index
)

# Apply mask: set invalid readings to NaN before averaging
values = aq_filtered[h_cols].copy()
values[~valid_mask] = np.nan

# Daily average is the mean of valid hourly readings for that day
aq_filtered = aq_filtered.copy()
aq_filtered["daily_avg_ugm3"] = values.mean(axis=1)

# Keep only the needed columns
aq_daily = aq_filtered[[
    "ESTACIO", "CODI_CONTAMINANT", "ANY", "MES", "DIA", "daily_avg_ugm3", "SOURCE_FILE"
]].copy()
aq_daily.columns = ["station_id","pollutant_code","year","month","day","daily_avg_ugm3","source_file"]

print(f"Daily average rows: {len(aq_daily):,}")
print(f"Rows with valid daily avg: {aq_daily['daily_avg_ugm3'].notna().sum():,}")


Daily average rows: 4,592
Rows with valid daily avg: 4,195


In [ ]:
# Join with the station metadata to get district
# Show what the stations metadata actually contains
stations_raw = con.execute("""
    SELECT CAST(Estacio AS VARCHAR) AS station_id,
           Nom_districte           AS district_raw,
           Nom_barri               AS neighborhood,
           CAST(Codi_Contaminant AS INT) AS pollutant_code_in_metadata
    FROM bronze.airquality_stations_raw
""").df()
print(f"Station metadata rows total: {len(stations_raw)}")
print(f"Unique station IDs in metadata: {stations_raw['station_id'].nunique()}")
print("Pollutant codes registered in station metadata:")
print(stations_raw.groupby("pollutant_code_in_metadata")["station_id"].nunique().to_string())

# One row per physical station, same station may appear once per pollutant
stations_unique = (
    stations_raw
    .drop_duplicates(subset=["station_id"])
    [["station_id", "district_raw", "neighborhood"]]
    .copy()
)
print(f"\nUnique stations after dedup: {len(stations_unique)}")

aq_daily["station_id"]       = aq_daily["station_id"].astype(str).str.strip()
stations_unique["station_id"] = stations_unique["station_id"].astype(str).str.strip()

aq_with_district = aq_daily.merge(
    stations_unique,
    on="station_id",
    how="left"
)

# Apply clean_district to the district name from metadata
aq_with_district["district"] = aq_with_district["district_raw"].apply(clean_district)

# Unmatched readings indicate station IDs present in readings but absent in metadata
unmatched = aq_with_district["district_raw"].isna().sum()
matched   = len(aq_with_district) - unmatched
print(f"\nJoin result: {matched:,} readings matched,  {unmatched:,} unmatched")
if unmatched > 0:
    bad_ids = sorted(aq_with_district.loc[aq_with_district["district_raw"].isna(),
                                          "station_id"].unique())
    print(f"Station IDs not in metadata: {bad_ids[:10]}")
    print("Possible causes: leading-zero mismatch (e.g. '7' vs '07'), station added")
    print(" after metadata was exported, or encoding difference in the CSV files.")
else:
    print("All station IDs resolved so no unmatched readings.")


Station metadata rows total: 55
Unique station IDs in metadata: 8
Pollutant codes registered in station metadata:
pollutant_code_in_metadata
1      4
6      4
7      8
8      8
9      3
10     6
12     8
14     6
101    1
106    1
107    1
108    1
109    1
110    1
112    1
114    1

Unique stations after dedup: 8

Join result: 4,592 readings matched,  0 unmatched
All station IDs resolved so no unmatched readings.


In [ ]:
# Aggregate to monthly district averages
aq_monthly = (
    aq_with_district
    .dropna(subset=["district", "daily_avg_ugm3"])
    .groupby(["district", "pollutant_code", "year", "month"], as_index=False)
    .agg(
        monthly_avg_ugm3 = ("daily_avg_ugm3", "mean"),
        days_measured    = ("daily_avg_ugm3", "count")
    )
)

# Add the pollutant name label
aq_monthly["pollutant_name"] = aq_monthly["pollutant_code"].map({8: "NO2", 10: "PM10"})

print(f"silver.airquality rows: {len(aq_monthly):,}")
pivot = aq_monthly.groupby(["district","pollutant_name"])["monthly_avg_ugm3"].mean().unstack().round(1)
display(pivot)

# If PM10 NaN still appears here after fixing the join, the cause is different: those districts genuinely have NO2 stations but NO PM10 sensor.
# That is a real infrastructure gap
no2_districts  = set(aq_monthly.loc[aq_monthly["pollutant_name"]=="NO2",  "district"].unique())
pm10_districts = set(aq_monthly.loc[aq_monthly["pollutant_name"]=="PM10", "district"].unique())
missing_pm10   = no2_districts - pm10_districts
no2_only_count = len(missing_pm10)

if missing_pm10:
    print(f"\nDistricts with NO2 monitoring but no PM10 sensor:")
    for d in sorted(missing_pm10):
        print(f"       {d}")
    print(" These are genuine sensor gaps, not a code error.")
    print("PM10 will be null in Silver and imputed with the city mean at the gold layer.")
else:
    print("\nPM10 coverage: all NO2 districts also have PM10 data,  no sensor gaps.")

print(f"\nNO2  districts covered ({len(no2_districts)}): {sorted(no2_districts)}")
print(f"PM10 districts covered ({len(pm10_districts)}): {sorted(pm10_districts)}")


silver.airquality rows: 143


pollutant_name,NO2,PM10
district,,
CIUTAT VELLA,22.4,NaN
EIXAMPLE,29.4,22.6
GRACIA,25.5,NaN
HORTA GUINARDO,18.6,17.5
LES CORTS,15.4,17.5
SANT MARTI,22.3,21.8
SANTS MONTJUIC,18.0,NaN
SARRIA SANT GERVASI,7.5,17.8



Districts with NO2 monitoring but no PM10 sensor:
       CIUTAT VELLA
       GRACIA
       SANTS MONTJUIC
 These are genuine sensor gaps, not a code error.
PM10 will be null in Silver and imputed with the city mean at the gold layer.

NO2  districts covered (8): ['CIUTAT VELLA', 'EIXAMPLE', 'GRACIA', 'HORTA GUINARDO', 'LES CORTS', 'SANT MARTI', 'SANTS MONTJUIC', 'SARRIA SANT GERVASI']
PM10 districts covered (5): ['EIXAMPLE', 'HORTA GUINARDO', 'LES CORTS', 'SANT MARTI', 'SARRIA SANT GERVASI']


In [ ]:
con.execute("CREATE OR REPLACE TABLE silver.airquality AS SELECT * FROM aq_monthly")
print(f"silver.airquality: {con.execute('SELECT COUNT(*) FROM silver.airquality').fetchone()[0]:,} rows")


silver.airquality: 143 rows


## Silver 4: EV charging stations
All 4 quarters loaded in Bronze:
 We will assign each station to a district by using Barcelona coordinate bounding boxes, and count the public charging points that each district has to give the most current picture of the EV infrastructure as it was at the end of 2025.


In [ ]:
# Assign districts using official polygon boundaries to get the correct correct districts
# The EV charging stations dataset contains longitude and latitude and no district column
# So the longitude and latitude coordinates should be assigned to a district
# To correctly assign the longitudes and latitudes to a district we use the dataset
# Source: BarcelonaCiutat_Districtes.json — Ajuntament de Barcelona Open Data

import requests
import pandas as pd
#%pip install shapely requests
from shapely.geometry import Point

In [ ]:
from shapely import wkt

In [ ]:
# Download official district polygons

DISTRICTS_URL = (
    "https://opendata-ajuntament.barcelona.cat/data/dataset/"
    "808daafa-d9ce-48c0-925a-fa5afdb1ed41/resource/"
    "5f8974a7-7937-4b50-acbc-89204d570df9/download"
)

print("Downloading official Barcelona district boundaries...")
response = requests.get(DISTRICTS_URL, timeout=30)
response.raise_for_status()
districts_raw = response.json()

print(f"Downloaded {len(districts_raw)} district records")
print(f"Fields: {list(districts_raw[0].keys())}")

Downloaded 10 district records
Fields: ['Codi_Districte', 'nom_districte', 'geometria_etrs89', 'geometria_wgs84']


In [ ]:
# Parse WKT polygons and build the lookup dict
# geometria_wgs84 contains a WKT string like:
#   POLYGON ((2.163 41.379, 2.164 41.380, ...))
# wkt.loads() converts this text into a Shapely polygon object that supports
# the .contains(point) method used below.
# Name cleaning matches the pipeline's CANONICAL_DISTRICTS convention:
#   "Sarrià-Sant Gervasi" → "SARRIA SANT GERVASI"

def clean_district_name(raw_name: str) -> str:
    """Standardise a district name to uppercase ASCII, matching pipeline convention."""
    name = raw_name.upper().strip()
    # Remove accents
    replacements = {
        "À": "A", "Á": "A", "È": "E", "É": "E",
        "Í": "I", "Ï": "I", "Ó": "O", "Ú": "U", "Ü": "U",
    }
    for accented, plain in replacements.items():
        name = name.replace(accented, plain)
    # Replace the hyphens with space, ex: Sants-Montjuïc becomes SANTS MONTJUIC)
    name = name.replace("-", " ")
    # Collapse multiple spaces
    name = " ".join(name.split())
    return name

# Build dict: clean_name becomes shapely polygon
district_polygons = {}
for record in districts_raw:
    name = clean_district_name(record["nom_districte"])
    polygon = wkt.loads(record["geometria_wgs84"])
    district_polygons[name] = polygon

print(f"\nPolygons loaded for {len(district_polygons)} districts:")
for name in sorted(district_polygons.keys()):
    print(f"  {name}")


Polygons loaded for 10 districts:
  CIUTAT VELLA
  EIXAMPLE
  GRACIA
  HORTA GUINARDO
  LES CORTS
  NOU BARRIS
  SANT ANDREU
  SANT MARTI
  SANTS MONTJUIC
  SARRIA SANT GERVASI


In [ ]:
# Print the official bounding boxes, replacing the manual district boxes we tried previously

print("\nOfficial bounding boxes from actual polygon geometry:")
print(f"{'District':<25} {'lat_min':>8} {'lat_max':>8} {'lon_min':>8} {'lon_max':>8}")
print("-" * 65)

# bounds:  minx, miny, maxx, maxy = lon_min, lat_min, lon_max, lat_max
for name in sorted(district_polygons.keys()):
    b = district_polygons[name].bounds
    lon_min, lat_min, lon_max, lat_max = b
    print(f"{name:<25} {lat_min:>8.4f} {lat_max:>8.4f} {lon_min:>8.4f} {lon_max:>8.4f}")


Official bounding boxes from actual polygon geometry:
District                   lat_min  lat_max  lon_min  lon_max
-----------------------------------------------------------------
CIUTAT VELLA               41.3577  41.3922   2.1631   2.1997
EIXAMPLE                   41.3750  41.4120   2.1423   2.1869
GRACIA                     41.3956  41.4249   2.1293   2.1720
HORTA GUINARDO             41.4069  41.4505   2.1204   2.1808
LES CORTS                  41.3757  41.4011   2.0978   2.1443
NOU BARRIS                 41.4255  41.4683   2.1557   2.1889
SANT ANDREU                41.4137  41.4621   2.1761   2.2108
SANT MARTI                 41.3834  41.4302   2.1755   2.2280
SANTS MONTJUIC             41.3170  41.3856   2.1005   2.1835
SARRIA SANT GERVASI        41.3901  41.4366   2.0523   2.1557


In [ ]:
# Point-in-polygon assignment function
# Point(longitude, latitude): longitude comes first in Shapely.
# This matches how WKT stores the coordinates: (x=longitude, y=latitude)
# .contains() returns True if the point is strictly inside the polygon.
# Points exactly on the the boundary return False for .contains()
# .within() also returns false for boundary points.
# So we use .covers() instead, which returns True for both inside and boundary.

def assign_district_polygon(lat, lon) -> str:
    if pd.isna(lat) or pd.isna(lon):
        return "UNKNOWN"

    try:
        point = Point(float(lon), float(lat))
    except (TypeError, ValueError):
        return "UNKNOWN"

    for district_name, polygon in district_polygons.items():
        if polygon.covers(point):
            return district_name

    return "UNKNOWN"

In [ ]:
# Load EV charging data and assign districts

ev_df = con.execute("SELECT * FROM bronze.ev_charging_raw").df()
print(f"Total rows across all quarters: {len(ev_df):,}")
print("Rows per quarter:")
print(ev_df["quarter"].value_counts().sort_index())
print(f"\nAll columns: {list(ev_df.columns)}")

Total rows across all quarters: 544
Rows per quarter:
quarter
Q1    137
Q2    138
Q3    136
Q4    133
Name: count, dtype: int64

All columns: ['station_id', 'quarter', 'network_name', 'latitude', 'longitude', 'address_str', 'postal_code', 'locality', 'access']


In [ ]:
# Identify the coordinate columns and apply assignment
# Check the column names, because there are different common variants in Barcelona open data:
#   latitud / longitud  (Catalan spelling, most common)
#   latitude / longitude
#   lat / lon

# Auto-detect lat/lon columns
def find_coord_column(df, candidates):
    for col in candidates:
        if col in df.columns:
            return col
    return None

LAT_COL = find_coord_column(ev_df, ["latitud", "latitude", "lat", "LATITUD"])
LON_COL = find_coord_column(ev_df, ["longitud", "longitude", "lon", "LONGITUD"])

if LAT_COL is None or LON_COL is None:
    print("Could not auto-detect coordinate columns.")
    print("Available columns:", list(ev_df.columns))
    print("Set LAT_COL and LON_COL manually below.")
else:
    print(f"Using: LAT_COL='{LAT_COL}', LON_COL='{LON_COL}'")
    print(ev_df[[LAT_COL, LON_COL]].describe())

Using: LAT_COL='latitude', LON_COL='longitude'
         latitude   longitude
count  544.000000  544.000000
mean    41.400882    2.163705
std      0.021341    0.027701
min     41.329694    2.107238
25%     41.387108    2.142061
50%     41.398553    2.164265
75%     41.411564    2.185633
max     41.459745    2.221153


In [ ]:


# Ensure numeric
ev_df[LAT_COL] = pd.to_numeric(ev_df[LAT_COL], errors="coerce")
ev_df[LON_COL] = pd.to_numeric(ev_df[LON_COL], errors="coerce")

# Apply polygon assignment

ev_df["district_assigned"] = ev_df.apply(
    lambda row: assign_district_polygon(row[LAT_COL], row[LON_COL]),
    axis=1
)

In [ ]:
# Validation
print("Spatial assignment validation report:")
print("=" * 60)

# Distribution across the different districts
print("\n[1] Chargers per district:")
dist_counts = ev_df["district_assigned"].value_counts()
print(dist_counts.to_string())

# Unmatched points
n_unknown = (ev_df["district_assigned"] == "UNKNOWN").sum()
pct_unknown = n_unknown / len(ev_df) * 100
print(f"\n[2] Unmatched (UNKNOWN): {n_unknown:,} rows ({pct_unknown:.1f}%)")

if n_unknown > 0:
    unknowns = ev_df[ev_df["district_assigned"] == "UNKNOWN"]
    print("    Sample unmatched coordinates:")
    print(unknowns[[LAT_COL, LON_COL]].head(5).to_string(index=False))

# Points outside Barcelona's geographic envelope
outside = ev_df[
    (ev_df[LAT_COL] < 41.32) | (ev_df[LAT_COL] > 41.48) |
    (ev_df[LON_COL] < 2.05)  | (ev_df[LON_COL] > 2.23)
]
print(f"\n[3] Points outside Barcelona bounds: {len(outside):,}")

# Missing coordinate rows
n_missing = ev_df[[LAT_COL, LON_COL]].isna().any(axis=1).sum()
print(f"\n[4] Rows with missing coordinates: {n_missing:,}")

# Are all of the 10 expected districts present?
expected = set(district_polygons.keys())
found    = set(ev_df["district_assigned"]) - {"UNKNOWN"}
missing  = expected - found
print(f"\n[5] Districts with no chargers assigned: {missing or 'None — all districts covered'}")

# Coverage assertion
pct_matched = (len(ev_df) - n_unknown) / len(ev_df) * 100
print(f"\n[6] Overall match rate: {pct_matched:.1f}%")
assert pct_matched >= 85, f"FAIL: Only {pct_matched:.1f}% matched — check coordinate columns"
print("    PASS: Match rate >= 85%")

Spatial assignment validation report:

[1] Chargers per district:
district_assigned
EIXAMPLE               89
SANT MARTI             79
SARRIA SANT GERVASI    76
SANT ANDREU            52
LES CORTS              50
SANTS MONTJUIC         48
CIUTAT VELLA           48
HORTA GUINARDO         38
GRACIA                 36
NOU BARRIS             24
UNKNOWN                 4

[2] Unmatched (UNKNOWN): 4 rows (0.7%)
    Sample unmatched coordinates:
 latitude  longitude
41.413142   2.221153
41.413142   2.221153
41.413142   2.221153
41.413142   2.221153

[3] Points outside Barcelona bounds: 0

[4] Rows with missing coordinates: 0

[5] Districts with no chargers assigned: None — all districts covered

[6] Overall match rate: 99.3%
    PASS: Match rate >= 85%


# Silver: EV Charging Stations
 Goal is one row per district with a count of active public charging points.

Deduplicate by station_id to keep the latest quarter
- A station present in Q1 and Q4 is the same physical station, counting it
- twice would inflate the totals. We keep the latest quarter so the snapshot
- reflects the current infrastructure state (end of 2025).

We filter to public access only
- Private chargers (workplace, residential) are not accessible to the general
- public and should not be counted as urban EV infrastructure supply.

Drop missing coordinates before district assignment:
- Point-in-polygon fails silently on NaN coords and returns UNKNOWN,
- polluting the district counts with unlocatable stations.

In [ ]:

ev_raw = con.execute("SELECT * FROM bronze.ev_charging_raw").df()

# Schema standardisation: cast and rename to snake_case
ev_raw["latitude"]  = pd.to_numeric(ev_raw[LAT_COL], errors="coerce")
ev_raw["longitude"] = pd.to_numeric(ev_raw[LON_COL], errors="coerce")
ev_raw["access"]    = ev_raw["access"].str.strip().str.upper()
ev_raw["quarter"]   = ev_raw["quarter"].str.strip().str.upper()

# Drop rows with no station_id or no coordinates
n_before = len(ev_raw)
ev_clean = ev_raw.dropna(subset=["station_id", "latitude", "longitude"]).copy()
print(f"Dropped {n_before - len(ev_clean):,} rows with missing station_id or coordinates")

# Deduplicate: keep the latest quarter per station
ev_clean = (
    ev_clean
    .sort_values("quarter")
    .drop_duplicates(subset="station_id", keep="last")
)
print(f"Unique stations after deduplication: {len(ev_clean):,}")
print(f"  Q4 snapshot: {(ev_clean['quarter'] == 'Q4').sum():,} stations")
print(f"  Earlier quarters (decommissioned by Q4): {(ev_clean['quarter'] != 'Q4').sum():,} stations")

# District assignment via official polygon boundaries
print("\nAssigning districts via point-in-polygon...")
ev_clean["district"] = ev_clean.apply(
    lambda r: assign_district_polygon(r["latitude"], r["longitude"]), axis=1
)

# Validation
n_unknown  = (ev_clean["district"] == "UNKNOWN").sum()
pct_matched = (len(ev_clean) - n_unknown) / len(ev_clean) * 100
print(f"Match rate: {pct_matched:.1f}%  |  UNKNOWN: {n_unknown:,}")
assert pct_matched >= 85, f"FAIL: only {pct_matched:.1f}% matched — check coordinate columns"

# Aggregate: public charging points per district
ev_silver = (
    ev_clean[ev_clean["access"] == "PUBLIC"]
    .groupby("district", as_index=False)
    .agg(
        ev_charging_points=("station_id", "count"),
        pct_from_q4=("quarter", lambda x: round(100 * (x == "Q4").sum() / len(x), 1))
    )
    .sort_values("ev_charging_points", ascending=False)
    .reset_index(drop=True)
)

# Write to Silver
con.execute("CREATE OR REPLACE TABLE silver.ev_charging AS SELECT * FROM ev_silver")
n = con.execute("SELECT COUNT(*) FROM silver.ev_charging").fetchone()[0]
print(f"\nsilver.ev_charging: {n} district rows")
display(ev_silver)


Dropped 0 rows with missing station_id or coordinates
Unique stations after deduplication: 138
  Q4 snapshot: 133 stations
  Earlier quarters (decommissioned by Q4): 5 stations

Assigning districts via point-in-polygon...
Match rate: 99.3%  |  UNKNOWN: 1

silver.ev_charging: 11 district rows


,district,ev_charging_points,pct_from_q4
0,EIXAMPLE,23,91.3
1,SANT MARTI,20,95.0
2,SARRIA SANT GERVASI,19,100.0
3,LES CORTS,13,92.3
4,SANT ANDREU,13,100.0
5,SANTS MONTJUIC,12,100.0
6,CIUTAT VELLA,12,100.0
7,HORTA GUINARDO,10,90.0
8,GRACIA,9,100.0
9,NOU BARRIS,6,100.0


## Silver 5: Vehicle Motorization Index

Filter for 'Turismes' (private cars) and aggregate to district level.

**Note:** 4 districts (Ciutat Vella, Gràcia, Les Corts, Sarrià-Sant Gervasi) have all-NA census sections
in the source portal data — this is a genuine upstream data gap, not a code error.
We impute these with the Barcelona city average and flag them in the data catalogue.

In [ ]:
# Pull the rows that have actual numeric values (skip NA rows)
vehicles_df = con.execute("""
    SELECT
        clean_district(Nom_Districte)                      AS district,
        CAST("Any" AS INT)                                 AS year,
        Tipus_Vehicle                                      AS vehicle_type,
        TRY_CAST(
            NULLIF(TRIM(CAST(Index_Motoritzacio AS VARCHAR)), 'NA')
        AS DOUBLE)                                         AS index_value
    FROM bronze.vehicles_raw
    WHERE Nom_Districte IS NOT NULL
      AND TRIM(CAST(Nom_Districte AS VARCHAR)) NOT IN ('', 'nan')
      AND Tipus_Vehicle IS NOT NULL
      AND TRIM(CAST(Index_Motoritzacio AS VARCHAR)) NOT IN ('NA', '', 'nan')
""").df()

print(f"Rows with actual index values: {len(vehicles_df):,}")

# Aggregate to district level, mean across the  census sections that have data
vehicles_agg = (
    vehicles_df
    .dropna(subset=["index_value"])
    .groupby(["district","year","vehicle_type"], as_index=False)
    .agg(
        avg_motorization_index     = ("index_value", "mean"),
        census_sections_with_data  = ("index_value", "count")
    )
)
vehicles_agg["avg_motorization_index"] = vehicles_agg["avg_motorization_index"].round(1)
print(f"Districts with measured data: {vehicles_agg['district'].nunique()}")


Rows with actual index values: 1,094
Districts with measured data: 6


In [ ]:
# Impute NaN districts with city average
# Affected: Ciutat Vella, Gràcia, Les Corts, Sarrià-Sant Gervasi
# since their census section range has no index values in the 2025 portal export

existing_silver = set(
    con.execute("""
        SELECT DISTINCT district FROM silver.complaints WHERE district != 'UNKNOWN'
        UNION
        SELECT DISTINCT district FROM silver.accidents  WHERE district != 'UNKNOWN'
    """).df()["district"].tolist()
)

# Barcelona always has exactly 10 administrative districts.
# We hardcode them so imputation still works even if complaints/accidents data is sparse.
CANONICAL_DISTRICTS = {
    "CIUTAT VELLA", "EIXAMPLE", "SANTS MONTJUIC", "LES CORTS",
    "SARRIA SANT GERVASI", "GRACIA", "HORTA GUINARDO",
    "NOU BARRIS", "SANT ANDREU", "SANT MARTI"
}

all_districts = existing_silver | CANONICAL_DISTRICTS
if len(all_districts) != 10:
    print(f"Expected 10 canonical districts, found {len(all_districts)}: {sorted(all_districts)}")
    print(" Check that clean_district() produces the exact spellings in CANONICAL_DISTRICTS.")
else:
    print(f"District reference list: {len(all_districts)} districts confirmed")

year_val  = int(vehicles_agg["year"].mode()[0])
all_types = vehicles_agg["vehicle_type"].unique().tolist()

imputed_rows = []
for vtype in all_types:
    city_mean = round(
        vehicles_agg.loc[vehicles_agg["vehicle_type"]==vtype, "avg_motorization_index"].mean(), 1
    )
    districts_with_data = set(
        vehicles_agg.loc[vehicles_agg["vehicle_type"]==vtype, "district"]
    )
    missing = all_districts - districts_with_data
    for d in sorted(missing):
        imputed_rows.append({
            "district": d, "year": year_val,
            "vehicle_type": vtype,
            "avg_motorization_index": city_mean,
            "census_sections_with_data": 0   # 0 = imputed, not measured
        })
        if vtype == "Turismes":
            print(f"Imputed {d}: Turismes index → {city_mean} (city average)")

import pandas as pd
vehicles_final = pd.concat(
    [vehicles_agg, pd.DataFrame(imputed_rows)], ignore_index=True
).sort_values(["district","vehicle_type"]).reset_index(drop=True)

con.execute("CREATE OR REPLACE TABLE silver.vehicles AS SELECT * FROM vehicles_final")
print(f"\nsilver.vehicles: {con.execute('SELECT COUNT(*) FROM silver.vehicles').fetchone()[0]:,} rows")

print("\nTurismes (private cars) per district  = imputed with city average:")
display(con.execute("""
    SELECT district,
           ROUND(avg_motorization_index,1) AS cars_per_1000,
           CASE WHEN census_sections_with_data = 0
                THEN ' imputed (city avg)' ELSE 'measured' END AS data_source
    FROM silver.vehicles
    WHERE vehicle_type = 'Turismes'
    ORDER BY district
""").df())


District reference list: 10 districts confirmed
Imputed CIUTAT VELLA: Turismes index → 271.9 (city average)
Imputed GRACIA: Turismes index → 271.9 (city average)
Imputed LES CORTS: Turismes index → 271.9 (city average)
Imputed SARRIA SANT GERVASI: Turismes index → 271.9 (city average)

silver.vehicles: 60 rows

Turismes (private cars) per district  = imputed with city average:


,district,cars_per_1000,data_source
0,CIUTAT VELLA,271.9,imputed (city avg)
1,EIXAMPLE,267.1,measured
2,GRACIA,271.9,imputed (city avg)
3,HORTA GUINARDO,279.4,measured
4,LES CORTS,271.9,imputed (city avg)
5,NOU BARRIS,252.3,measured
6,SANT ANDREU,296.7,measured
7,SANT MARTI,302.1,measured
8,SANTS MONTJUIC,233.8,measured
9,SARRIA SANT GERVASI,271.9,imputed (city avg)


In [ ]:
print("=== DISTRICT NAME CONSISTENCY CHECK ===\n")

complaints_d = set(con.execute("SELECT DISTINCT district FROM silver.complaints WHERE district != 'UNKNOWN'").df()["district"])
accidents_d  = set(con.execute("SELECT DISTINCT district FROM silver.accidents  WHERE district != 'UNKNOWN'").df()["district"])
airquality_d = set(con.execute("SELECT DISTINCT district FROM silver.airquality WHERE district != 'UNKNOWN'").df()["district"])
ev_d         = set(con.execute("SELECT DISTINCT district FROM silver.ev_charging WHERE district != 'UNKNOWN'").df()["district"])
vehicles_d   = set(con.execute("SELECT DISTINCT district FROM silver.vehicles   WHERE district != 'UNKNOWN'").df()["district"])

# Known legitimate gaps: no air quality stations in these 2 districts
KNOWN_GAPS = {"airquality": {"NOU BARRIS", "SANT ANDREU"}}

all_districts = complaints_d | accidents_d | airquality_d | ev_d | vehicles_d
print(f"Checking {len(all_districts)} unique district names:\n")

naming_errors = []
for d in sorted(all_districts):
    results = {
        "complaints" : d in complaints_d,
        "accidents"  : d in accidents_d,
        "airquality" : d in airquality_d,
        "ev"         : d in ev_d,
        "vehicles"   : d in vehicles_d,
    }
    row = f"  {d:<30}"
    has_error = False
    for col, present in results.items():
        is_known_gap = (d in KNOWN_GAPS.get(col, set()))
        if present:
            row += f" {col}:✅"
        elif is_known_gap:
            row += f" {col}:⚪"   # known gap — expected
        else:
            row += f" {col}:❌"   # real problem
            has_error = True
    if has_error:
        naming_errors.append(d)
    print(row)

print("\nLegend: ✅ present  ⚪ known gap (no station)  ❌ naming mismatch")
if naming_errors:
    print(f"\n⚠️  NAMING MISMATCHES TO FIX: {naming_errors}")
    print("   Fix clean_district() above and re-run Silver from the top.")
else:
    print("\n✅ All naming issues resolved — safe to run 03_gold.ipynb")


=== DISTRICT NAME CONSISTENCY CHECK ===

Checking 10 unique district names:

  CIUTAT VELLA                   complaints:✅ accidents:✅ airquality:✅ ev:✅ vehicles:✅
  EIXAMPLE                       complaints:✅ accidents:✅ airquality:✅ ev:✅ vehicles:✅
  GRACIA                         complaints:✅ accidents:✅ airquality:✅ ev:✅ vehicles:✅
  HORTA GUINARDO                 complaints:✅ accidents:✅ airquality:✅ ev:✅ vehicles:✅
  LES CORTS                      complaints:✅ accidents:✅ airquality:✅ ev:✅ vehicles:✅
  NOU BARRIS                     complaints:✅ accidents:✅ airquality:⚪ ev:✅ vehicles:✅
  SANT ANDREU                    complaints:✅ accidents:✅ airquality:⚪ ev:✅ vehicles:✅
  SANT MARTI                     complaints:✅ accidents:✅ airquality:✅ ev:✅ vehicles:✅
  SANTS MONTJUIC                 complaints:✅ accidents:✅ airquality:✅ ev:✅ vehicles:✅
  SARRIA SANT GERVASI            complaints:✅ accidents:✅ airquality:✅ ev:✅ vehicles:✅

Legend: ✅ present  ⚪ known gap (no station)  ❌ namin

In [ ]:
print("\nSILVER LAYER COMPLETE")
tables_to_check = ["complaints","complaints_masked","accidents","airquality","ev_charging","vehicles"]
for t in tables_to_check:
    try:
        n = con.execute(f"SELECT COUNT(*) FROM silver.{t}").fetchone()[0]
        print(f"  silver.{t:25s}: {n:>8,} rows")
    except Exception:
        print(f"  silver.{t:25s}: NOT YET CREATED")
print("\nNext step is to run 03_gold.ipynb")
print("Gold reads from silver.complaints_masked (PII-masked version).")



SILVER LAYER COMPLETE
  silver.complaints               :  287,304 rows
  silver.complaints_masked        :  287,304 rows
  silver.accidents                :    8,072 rows
  silver.airquality               :      143 rows
  silver.ev_charging              :       11 rows
  silver.vehicles                 :       60 rows

Next step is to run 03_gold.ipynb
Gold reads from silver.complaints_masked (PII-masked version).


In [ ]:
# close the connection before opening gold
con.close()
print("Connection closed — barcelona_urban.db is now free.")
print("03_gold.ipynb can be safely opened")


Connection closed — barcelona_urban.db is now free.
03_gold.ipynb can be safely opened
